In [0]:
-- Query 1: Executive Revenue Dashboard
-- Monthly GMV trend with month-over-month growth
WITH monthly AS (
    SELECT 
        date_format(order_purchase_timestamp, 'yyyy-MM') AS month,
        ROUND(SUM(oi.price), 2) AS gmv,
        COUNT(DISTINCT o.order_id) AS orders
    FROM shopsphere_catalog.retail.orders o
    JOIN shopsphere_catalog.retail.order_items oi USING (order_id)
    WHERE o.order_status = 'DELIVERED'
    GROUP BY month
    ORDER BY month
)
SELECT 
    month,
    gmv,
    orders,
    ROUND(LAG(gmv) OVER (ORDER BY month), 2) AS prev_month_gmv,
    ROUND((gmv - LAG(gmv) OVER (ORDER BY month)) / 
          LAG(gmv) OVER (ORDER BY month) * 100, 1) AS mom_growth_pct
FROM monthly;


-- Query 2: Top Product Categories by Revenue
SELECT 
    COALESCE(p.product_category_name, 'Unknown') AS category,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    ROUND(SUM(oi.price), 0) AS total_revenue_brl,
    ROUND(AVG(oi.price), 2) AS avg_item_price,
    ROUND(SUM(oi.price) * 100.0 / SUM(SUM(oi.price)) OVER (), 2) AS revenue_share_pct
FROM shopsphere_catalog.retail.order_items oi
JOIN shopsphere_catalog.retail.products p ON oi.product_id = p.product_id
JOIN shopsphere_catalog.retail.orders o ON oi.order_id = o.order_id
WHERE o.order_status = 'DELIVERED'
GROUP BY category
ORDER BY total_revenue_brl DESC
LIMIT 15;

-- Query 3: Customer Retention Analysis (Repeat Buyers)
SELECT
  order_count_bucket,
  COUNT(*) AS customer_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_customers
FROM ( SELECT
      customer_id,
      CASE
        WHEN COUNT(*) = 1 THEN '1 order'
        WHEN COUNT(*) BETWEEN 2 AND 3 THEN '2-3 orders'
        WHEN COUNT(*) BETWEEN 4 AND 5 THEN '4-5 orders'
        ELSE '6+ orders' END AS order_count_bucket FROM
      shopsphere_catalog.retail.orders WHERE
      order_status = 'DELIVERED' GROUP BY
      customer_id
  ) sub
GROUP BY order_count_bucket
ORDER BY CASE order_count_bucket
    WHEN '1 order' THEN 1
    WHEN '2-3 orders' THEN 2
    WHEN '4-5 orders' THEN 3 ELSE 4
  END;
